# Поиск услуг: кандидатогенерация

Возвращаем до 50 уникальных объявлений на запрос. Метрика — **macro Recall@50**: средняя доля найденных позитивов по запросам.

Выполнен обзор данных. Split, E0–E8 и `answer.csv` пока не готовы.
Исходные данные не изменяем; итоговые запросы не используем для настройки.

## 1. Настройки

Запуск из корня проекта: `.venv/bin/python scripts/run_notebook.py`.
Расчёты и результаты хранятся в notebook.

In [1]:
from pathlib import Path
import hashlib
import importlib.metadata
import platform
import random
import shutil
import subprocess

import numpy as np
import pandas as pd
import psutil
import pyarrow.parquet as pq
from IPython.display import display

DATA_DIR = Path("dataset")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 100)

## 2. Данные

Проверяем размеры, схемы, формат ID и уникальность ключей benchmark.
Для исходных файлов считаем SHA-256.

In [2]:
QUERY_FIELDS = [
    "search_query", "search_location_id", "search_is_delivery_search",
    "search_infm_params_text", "search_category",
]


def normalize_text(values: pd.Series) -> pd.Series:
    """Нижний регистр и нормализация пробелов."""
    return values.fillna("").str.lower().str.replace(r"\s+", " ", regex=True).str.strip()


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def resources(root: Path) -> dict:
    try:
        gpu = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
            capture_output=True, text=True, check=True, timeout=10,
        ).stdout.strip()
    except (OSError, subprocess.SubprocessError):
        gpu = "unavailable"
    return {
        "python": platform.python_version(), "platform": platform.platform(),
        "cpu_physical": psutil.cpu_count(logical=False), "cpu_logical": psutil.cpu_count(),
        "ram_total_bytes": psutil.virtual_memory().total,
        "ram_available_bytes": psutil.virtual_memory().available,
        "disk_free_bytes": shutil.disk_usage(root).free, "gpu": gpu,
        "versions": {name: importlib.metadata.version(name) for name in
                     ["pandas", "numpy", "pyarrow", "nbformat", "nbclient", "psutil"]},
    }

In [3]:
def audit_data(data_dir: Path) -> dict:
    tables, files, columns, examples = {}, [], [], {}
    for name in ["benchmark_items", "benchmark_queries", "train"]:
        path = data_dir / f"{name}.parquet"
        frame = pd.read_parquet(path)
        tables[name] = frame
        files.append({"file": path.name, "bytes": path.stat().st_size,
                      "rows": len(frame), "columns": len(frame.columns), "sha256": sha256(path)})
        schema = pq.read_schema(path)
        for column in frame:
            values = frame[column]
            columns.append({"table": name, "column": column, "type": str(schema.field(column).type),
                            "nulls": int(values.isna().sum()), "unique": int(values.nunique()),
                            "empty_strings": int(values.eq("").sum())})
        examples[name] = frame.head(3).map(
            lambda x: str(x)[:240] if pd.notna(x) else None)

    items, queries, train = (tables[n] for n in ["benchmark_items", "benchmark_queries", "train"])
    for frame, key, pattern in [(items, "item_id", r"[0-9a-f]{16}"),
                                (train, "item_id", r"[0-9a-f]{16}"),
                                (queries, "query_id", r".{16}")]:
        if not frame[key].str.fullmatch(pattern).fillna(False).all():
            raise ValueError(f"Invalid {key}")
    if not items.item_id.is_unique or not queries.query_id.is_unique:
        raise ValueError("Duplicate benchmark IDs require an explicit resolution")

    # В train нет query_id; группируем по тексту и контексту.
    identity = train[QUERY_FIELDS].copy()
    identity["search_query"] = normalize_text(identity.search_query)
    group_ids = identity.groupby(QUERY_FIELDS, dropna=False, sort=True).ngroup()
    pairs = pd.DataFrame({"group": group_ids, "item_id": train.item_id}).drop_duplicates()
    pairs["in_corpus"] = pairs.item_id.isin(items.item_id)
    positives = pairs.groupby("group").agg(positives=("item_id", "size"), in_corpus=("in_corpus", "sum"))
    contexts = identity.drop_duplicates().groupby("search_query", dropna=False).size()
    item_text = pd.DataFrame({field: normalize_text(items[field]) for field in
                              ["item_title_raw", "item_description_raw"]})
    title_counts = item_text.groupby("item_title_raw").size()
    summary = {
        "train_rows": len(train), "corpus_items": len(items), "benchmark_queries": len(queries),
        "train_has_query_id": "query_id" in train, "train_has_explicit_label": False,
        "label_assumption": "Each train row is a positive interaction; no explicit label is provided.",
        "query_context_groups": len(positives),
        "normalized_train_texts": int(identity.search_query.nunique()),
        "texts_with_multiple_contexts": int(contexts.gt(1).sum()),
        "duplicate_positive_rows_after_normalization": len(train) - len(pairs),
        "unique_positive_pairs": len(pairs),
        "positive_pairs_outside_corpus": int((~pairs.in_corpus).sum()),
        "unique_positive_items_outside_corpus": int(pairs.loc[~pairs.in_corpus, "item_id"].nunique()),
        "groups_without_retrievable_positive": int(positives.in_corpus.eq(0).sum()),
        "positive_count_distribution": positives.positives.describe(percentiles=[.5, .9, .99]).to_dict(),
        "macro_recall_corpus_ceiling": float((positives.in_corpus / positives.positives).mean()),
        "empty_train_query_rows": int(identity.search_query.eq("").sum()),
        "empty_benchmark_query_rows": int(normalize_text(queries.search_query).eq("").sum()),
        "benchmark_repeated_text_rows": int(normalize_text(queries.search_query).duplicated().sum()),
        "duplicate_item_text_rows": int(item_text.duplicated().sum()),
        "repeated_title_groups": int(title_counts.gt(1).sum()),
        "near_duplicate_status": "Not measured; repeated titles alone do not prove near duplication.",
        "event_time_columns": [c for c in train if any(t in c.lower() for t in ["timestamp", "date", "event_time"])],
        "dense_float32_384_bytes": len(items) * 384 * 4,
        "dense_float32_768_bytes": len(items) * 768 * 4,
    }
    return {
        "summary": summary, "files": pd.DataFrame(files),
        "columns": pd.DataFrame(columns), "examples": examples,
        "resources": resources(data_dir.parent),
    }

In [4]:
audit = audit_data(DATA_DIR)
summary = audit["summary"]
files = audit["files"]
files["Размер, МиБ"] = (files["bytes"] / 2**20).round(2)
display(files[["file", "rows", "columns", "Размер, МиБ"]].rename(
    columns={"file": "Файл", "rows": "Строки", "columns": "Поля"}))
display(files[["file", "sha256"]])

,Файл,Строки,Поля,"Размер, МиБ"
0,benchmark_items.parquet,189212,14,186.61
1,benchmark_queries.parquet,2452,6,0.12
2,train.parquet,497673,19,467.53


,file,sha256
0,benchmark_items.parquet,193b3a3961464620cbf8d8797152b7f90cae1826ca749fb7817a210984a09899
1,benchmark_queries.parquet,e49de4fb76f03979a4af96034817767d39ae5de9f94e254fed028243188dda06
2,train.parquet,e150ab7a5c98769f643b95ee3a02dc0f664b647facd69a7d280ec6d48ca554a7


`benchmark_items` — корпус кандидатов, `benchmark_queries` — итоговые запросы,
`train` — пары запрос–объявление. Ниже — типы полей, пропуски, уникальные значения и пустые строки.

In [5]:
columns = audit["columns"]
column_names = {"column": "Поле", "type": "Тип Parquet", "nulls": "Пропуски",
                "unique": "Уникальные значения", "empty_strings": "Пустые строки"}
for table in files["file"].str.removesuffix(".parquet"):
    print(table)
    display(columns.loc[columns["table"].eq(table)].drop(columns="table").rename(columns=column_names))

benchmark_items


,Поле,Тип Parquet,Пропуски,Уникальные значения,Пустые строки
0,item_title_raw,string,0,119952,0
1,item_rating_reviews_count,double,11615,1170,0
2,item_rating,double,17631,6007,0
3,item_price,"decimal128(27, 15)",0,2949,0
4,item_microcat_id,int64,0,752,0
5,item_longitude,"decimal128(18, 15)",1,120004,0
6,item_location_id,int64,0,2877,0
7,item_latitude,"decimal128(17, 15)",1,119778,0
8,item_is_phone_hidden,bool,0,2,0
9,item_is_message_forbidden,bool,0,2,0


benchmark_queries


,Поле,Тип Parquet,Пропуски,Уникальные значения,Пустые строки
14,query_id,large_string,0,2452,0
15,search_query,large_string,0,2452,0
16,search_location_id,int64,0,273,0
17,search_is_delivery_search,int32,0,1,0
18,search_infm_params_text,large_string,0,132,1546
19,search_category,int64,0,2,0


train


,Поле,Тип Parquet,Пропуски,Уникальные значения,Пустые строки
20,search_query,string,0,74529,0
21,search_location_id,int64,0,2782,0
22,search_is_delivery_search,int32,0,2,0
23,search_infm_params_text,string,0,3724,163999
24,search_category,int64,0,4,0
25,item_title_raw,string,0,210029,0
26,item_rating_reviews_count,double,19211,1372,0
27,item_rating,double,28232,8832,0
28,item_price,"decimal128(27, 15)",0,2992,0
29,item_microcat_id,int64,0,212,0


### Примеры

Для отображения обрезаем значения до 240 символов.

In [6]:
for table, title in [("benchmark_queries", "Итоговый запрос"),
                     ("benchmark_items", "Объявление из корпуса"),
                     ("train", "Строка обучающей разметки")]:
    examples = audit["examples"][table]
    print(title)
    display(examples.iloc[0].rename("Пример значения").to_frame())

Итоговый запрос


,Пример значения
query_id,70DfDUpwjxB4lzFd
search_query,перевозки владикавказ тбилиси
search_location_id,649820
search_is_delivery_search,0
search_infm_params_text,
search_category,114


Объявление из корпуса


,Пример значения
item_title_raw,Ремонт/выкуп компьют. и ноутбуков с выездом на дом
item_rating_reviews_count,57.0
item_rating,5.0
item_price,500.000000000000000
item_microcat_id,2097573
item_longitude,42.047193961004901
item_location_id,631060
item_latitude,44.228180450924000
item_is_phone_hidden,False
item_is_message_forbidden,False


Строка обучающей разметки


,Пример значения
search_query,скупка телевизоров
search_location_id,652430
search_is_delivery_search,0
search_infm_params_text,
search_category,114
item_title_raw,Скупка б/у техники
item_rating_reviews_count,91.0
item_rating,4.989010989010989
item_price,1.000000000000000
item_microcat_id,2303374


В train нет `query_id`, явного label и времени событий. Считаем каждую строку позитивным взаимодействием — это допущение по формату данных и постановке задачи. Временной split построить нельзя.

Есть заголовок, описание, категория, география, цена и текст атрибутов. Для первого BM25 используем только заголовок и описание; вклад metadata проверим отдельно в E7.

## 3. Как определить запрос и учесть повторы

Одинаковый текст может относиться к разным городам или фильтрам. Для учёта позитивов определяем запрос сочетанием нормализованного текста и всех пяти `search_*` полей. Нормализация — нижний регистр и схлопывание пробелов; числа, «ё» и короткие слова сохраняем. ID не нормализуем.

Для будущего split объединим все контексты одного нормализованного текста в одну группу. Это не даст одинаковому тексту оказаться одновременно в train и validation.

In [7]:
query_counts = {
    "Строки train": summary["train_rows"],
    "Различные нормализованные тексты": summary["normalized_train_texts"],
    "Запросы с учётом контекста": summary["query_context_groups"],
    "Тексты с несколькими контекстами": summary["texts_with_multiple_contexts"],
    "Повторные позитивные пары": summary["duplicate_positive_rows_after_normalization"],
    "Уникальные позитивные пары": summary["unique_positive_pairs"],
}
display(pd.Series(query_counts, name="Количество").to_frame())
distribution = summary["positive_count_distribution"]
display(pd.Series({"Минимум": distribution["min"], "Медиана": distribution["50%"],
                   "90-й перцентиль": distribution["90%"], "99-й перцентиль": distribution["99%"],
                   "Максимум": distribution["max"], "Среднее": distribution["mean"]},
                  name="Позитивов на запрос с контекстом").to_frame().round(2))

,Количество
Строки train,497673
Различные нормализованные тексты,73904
Запросы с учётом контекста,354313
Тексты с несколькими контекстами,27894
Повторные позитивные пары,30644
Уникальные позитивные пары,467029


,Позитивов на запрос с контекстом
Минимум,1.00
Медиана,1.00
90-й перцентиль,2.00
99-й перцентиль,6.00
Максимум,278.00
Среднее,1.32


Большинство запросов имеет один известный позитив, но есть запросы с сотнями позитивов. Все пары одного запроса должны попадать в одну часть split; повторные пары не должны увеличивать вес объявления в метрике.

## 4. Можно ли найти размеченные объявления в корпусе

Это проверка достижимости метрики до запуска моделей. Для каждой пары проверяем наличие `item_id` в `benchmark_items`. Позитивы вне корпуса сохраняем в знаменателе: их исключение без оговорки завысило бы оценку.

In [8]:
total = summary["unique_positive_pairs"]
outside = summary["positive_pairs_outside_corpus"]
coverage = pd.DataFrame({
    "Позитивные пары": [total - outside, outside],
}, index=["Объявление есть в корпусе", "Объявления нет в корпусе"])
coverage["Доля пар, %"] = (100 * coverage["Позитивные пары"] / total).round(2)
display(coverage)
display(pd.Series({
    "Запросы с контекстом": summary["query_context_groups"],
    "Из них без доступных позитивов": summary["groups_without_retrievable_positive"],
}, name="Количество").to_frame())
print(f"Верхняя граница macro Recall по доступности корпуса: {summary['macro_recall_corpus_ceiling']:.2%}")

,Позитивные пары,"Доля пар, %"
Объявление есть в корпусе,29571,6.33
Объявления нет в корпусе,437458,93.67


,Количество
Запросы с контекстом,354313
Из них без доступных позитивов,327764


Верхняя граница macro Recall по доступности корпуса: 6.08%


Вне корпуса оказались 437 458 из 467 029 позитивных пар. При текущем определении запроса верхняя граница macro Recall по доступности корпуса — **6,08%**. Ограничение выдачи до 50 может снизить её ещё сильнее. Это свойство всех train-групп, а не оценка модели или будущего validation.

В основной метрике запросы, чьи позитивы полностью вне корпуса, получат ноль. Дополнительно будем показывать метрику по доступным позитивам с явным указанием знаменателя. Запросы без известных позитивов вообще не имеют определённого Recall: их нужно учитывать отдельным счётчиком.

## 5. Дубли объявлений и пустые запросы

Проверяем точные совпадения нормализованного заголовка и описания. Они могут облегчать перенос между частями split, поэтому при дальнейшем анализе нужно учитывать дубли объявлений. Само совпадение заголовка ещё не означает, что объявления одинаковы.

In [9]:
display(pd.Series({
    "Повторные строки title + description в корпусе": summary["duplicate_item_text_rows"],
    "Группы повторяющихся заголовков": summary["repeated_title_groups"],
    "Пустые запросы в train": summary["empty_train_query_rows"],
    "Пустые итоговые запросы": summary["empty_benchmark_query_rows"],
    "Повторные тексты итоговых запросов": summary["benchmark_repeated_text_rows"],
}, name="Количество").to_frame())

,Количество
Повторные строки title + description в корпусе,19097
Группы повторяющихся заголовков,15195
Пустые запросы в train,0
Пустые итоговые запросы,0
Повторные тексты итоговых запросов,1


Найдено 19 097 повторных строк сочетания заголовка и описания при разных item_id. Объявления не удаляем: метрика проверяет конкретные ID. Близкие, но не идентичные копии пока не измерены. Проверки уникальности и формата benchmark ID прошли; пустых поисковых текстов не найдено.

## 6. Ресурсы и предварительный бюджет

Проверяем доступную память и оборудование, чтобы выбрать размер dense-модели. Ниже — снимок ресурсов при запуске, а не измерение потребления retrieval. Наличие GPU в `nvidia-smi` ещё не подтверждает работоспособность PyTorch/CUDA.

In [10]:
resource_info = audit["resources"]
display(pd.Series({
    "Python": resource_info["python"],
    "CPU: физические ядра / потоки": f"{resource_info['cpu_physical']} / {resource_info['cpu_logical']}",
    "RAM всего, ГиБ": round(resource_info["ram_total_bytes"] / 2**30, 1),
    "RAM доступно, ГиБ": round(resource_info["ram_available_bytes"] / 2**30, 1),
    "Диск свободно, ГиБ": round(resource_info["disk_free_bytes"] / 2**30, 1),
    "GPU / память, МиБ / драйвер": resource_info["gpu"],
}, name="Значение").to_frame())
display(pd.DataFrame({
    "Размерность": [384, 768],
    "Embeddings float32, МиБ": [round(summary[f"dense_float32_{d}_bytes"] / 2**20, 1) for d in [384, 768]],
}))
display(pd.Series(resource_info["versions"], name="Версия библиотеки").to_frame())

,Значение
Python,3.12.3
CPU: физические ядра / потоки,16 / 32
"RAM всего, ГиБ",60.4
"RAM доступно, ГиБ",45.5
"Диск свободно, ГиБ",1315.5
"GPU / память, МиБ / драйвер","NVIDIA GeForce RTX 5090, 32607 MiB, 595.91.07"


,Размерность,"Embeddings float32, МиБ"
0,384,277.2
1,768,554.3


,Версия библиотеки
pandas,3.0.6
numpy,2.5.3
pyarrow,25.0.1
nbformat,5.11.1
nbclient,0.11.0
psutil,7.2.2


Массив embeddings потребует примерно 277–554 МиБ при размерности 384–768. Эта оценка не включает модель, батчи и индекс. Время индексации и обучения ещё не измерено.

Предварительный бюджет до настройки:

| Этап | Объём проверки |
| --- | --- |
| E0 | Один BM25 baseline без подбора |
| E1 | До пяти вариантов полей и весов |
| E2 | Word и character TF-IDF |
| E3 | Одна zero-shot embedding-модель |
| E4 | До трёх RRF-конфигураций |
| E5–E6 | Бюджет уточним после измерения скорости dense inference |

Все методы сравним на одном сохранённом split.

## 7. Следующий этап

Обзор выполнен на полных файлах. Большинство позитивов отсутствует в корпусе; учитываем их в знаменателе основной метрики.

Далее: split по нормализованному тексту (20% групп, seed 42), macro Recall@K и BM25 E0. Проверим дедупликацию top-k, соответствие строк `item_id` и метрики на примерах с известным ответом.

Ещё не выполнены: split, анализ близких копий, E0–E8, проверка GPU для обучения и submission.